# SpiceyPy scratchpad

A notebook for quick SpiceyPy one-offs, right in the browser. Run the two setup cells below first: the first installs NumPy, SciPy, SymPy, Astropy, matplotlib and SpiceyPy (the first run downloads Pyodide and the packages, so it can take a little while), the second loads a few generic kernels from the `scratchpad` folder: leap seconds, planetary constants and a planetary ephemeris for 2000-2019.

Notebooks are saved in your browser's storage, so use *File > Download* to keep a copy.

In [ ]:
%pip install numpy scipy sympy astropy matplotlib spiceypy

In [ ]:
import numpy as np
import scipy
import sympy
import astropy
import astropy.units as u
from astropy.time import Time
import matplotlib
import matplotlib.pyplot as plt
import spiceypy as spice

spice.furnsh(
    [
        "scratchpad/naif0012.tls",  # leap seconds
        "scratchpad/pck00010.tpc",  # body radii and orientation
        "scratchpad/gm_de431.tpc",  # body GM values
        "scratchpad/de440s_2000_to_2020_simplified.bsp",  # planet barycenters
    ]
)
print(f"SpiceyPy {spice.__version__} ({spice.tkvrsn('TOOLKIT')})")
print(f"NumPy {np.__version__}, SciPy {scipy.__version__}, SymPy {sympy.__version__}")
print(f"Astropy {astropy.__version__}, matplotlib {matplotlib.__version__}")
print(f"Loaded {spice.ktotal('ALL')} kernels")

## A starter plot

The inner planets' orbits around the solar system barycenter during 2019, in the ecliptic plane.

In [ ]:
ets = np.linspace(spice.str2et("2019-01-01"), spice.str2et("2019-12-31"), 1000)
fig, ax = plt.subplots(figsize=(6, 6))
for planet in ["MERCURY", "VENUS", "EARTH", "MARS"]:
    pos, _ = spice.spkpos(f"{planet} BARYCENTER", ets, "ECLIPJ2000", "NONE", "0")
    au = pos / spice.convrt(1, "AU", "KM")
    ax.plot(au[:, 0], au[:, 1], label=planet.title())
    ax.plot(au[0, 0], au[0, 1], "o", color=ax.lines[-1].get_color())
ax.set_aspect("equal")
ax.set_xlabel("x (AU)")
ax.set_ylabel("y (AU)")
ax.set_title("Inner planets during 2019 (dots: Jan 1)")
ax.legend(loc="upper right")
plt.show()

## Your turn

This cell is yours: replace it with anything SpiceyPy related. A few ideas: where Mars was on your birthday (if it was in 2000-2019), Saturn's distance from Earth over the years, or when Venus and Mars line up as seen from Earth (`spice.gfsep`). Astropy times convert with `spice.str2et(Time(...).isot)`.

In [ ]:
et = spice.str2et(Time("2019-06-10", scale="utc").isot)
# Jupiter was at opposition, closest to Earth for the year, on 2019-06-10
pos, lt = spice.spkpos("JUPITER BARYCENTER", et, "J2000", "LT+S", "EARTH")
print(f"On {spice.et2utc(et, 'C', 0)}, light took {lt / 60:.1f} minutes from Jupiter")